In [ ]:
#packages
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.dataframe
import pyspark.sql.functions as f
from pyspark.sql.functions import col
import math as m

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("Project 1")
    
    # === MEMORY MANAGEMENT ===
    .config("spark.driver.memory", "4g")          # Increase driver memory (safe for 16GB+ systems)
    .config("spark.executor.memory", "4g")        # Executors share same JVM locally
    .config("spark.driver.maxResultSize", "2g")   # Prevent large collect() results crashing driver

    # === PARALLELISM & SHUFFLING ===
    .config("spark.sql.shuffle.partitions", "48")  # Default is 200 — too high locally
    .config("spark.default.parallelism", "8")      # ~ number of cores on your system
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # Optimal shuffle partition size

    # === PERFORMANCE TUNING ===
    .config("spark.memory.fraction", "0.85")        # 85% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.3")  # 30% of execution memory for caching
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # Fast pandas conversion

    # === TEMP STORAGE ===
    .config("spark.local.dir", "/tmp/spark-temp")   # Disk spill location for large shuffles

    # === DEFAULT OPTIONS ===
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", True)
    .config("spark.sql.session.timeZone", "Etc/UTC")

    .getOrCreate()
)

In [ ]:
#reading in data

tbl_merchants_raw = spark.read.parquet('../data/tables/merchant_data/tbl_merchants.parquet')
consumer_user_details = spark.read.parquet('../data/tables/merchant_data/consumer_user_details.parquet')
transactions21 = spark.read.parquet('../data/tables/transaction_data/transactions_20210228_20210827_snapshot/')
transactions2122 = spark.read.parquet('../data/tables/transaction_data/transactions_20210828_20220227_snapshot/')
transactions22 = spark.read.parquet('../data/tables/transaction_data/transactions_20220228_20220828_snapshot/')
con_fraud_prob = spark.read.option("header","true").csv('../data/tables/merchant_data/consumer_fraud_probability.csv')
merch_fraud_prob = spark.read.option("header", "true").csv('../data/tables/merchant_data/merchant_fraud_probability.csv')

tbl_consumer_raw = spark.read.option("header", "true").csv('../data/tables/merchant_data/tbl_consumer.csv')

transactions = transactions21.unionByName(transactions2122)
transactions = transactions.unionByName(transactions22)

In [ ]:
# Load first CSV
postcodes_df = spark.read.csv("../data/income/2024 Locality to 2021 SA2 Coding Index.csv", header=True, inferSchema=True)

# Load second CSV
income_df = spark.read.csv("../data/income/sa2_income.csv", header=True, inferSchema=True)

In [ ]:
#Functions
def find_NULL(dfs):

    """Finds any rows with NULLs over different datasets"""

    for df in dfs:
        condition = f.lit(False)
        for col_name in df.columns:
            condition = condition | f.col(col_name).isNull()

        df.filter(condition).show()
    return df.filter(condition).count()

def filter_outliers(data, variables):
    
    """filters outliers of continuous data"""

    n=data.count()
    for feature in variables:
        # Calculate Q1 and Q3
        quantiles = data.approxQuantile(feature, [0.25, 0.75], 0.01)
        q1, q3 = quantiles
        iqr = q3 - q1

        #from ADS lecture slides, n>>100
        scale = m.sqrt(m.log(n)) - 0.5
        if scale<3:
            scale=3
        lower_bound = q1 - scale * iqr
        upper_bound = q3 + scale * iqr
        if lower_bound<0:
            data = data.filter((col(feature) >= 0) & (col(feature) <= upper_bound))
        else:
            data = data.filter((col(feature) >= lower_bound) & (col(feature) <= upper_bound))
    
    return data

def spark_shape(self):
    
    """Easy function for shape of a spark df"""

    return (self.count(), len(self.columns))

pyspark.sql.dataframe.DataFrame.shape = property(spark_shape)

In [ ]:
#cleaning tags
string = "name|address|state|postcode|gender|consumer_id"

# Clean consumer table
tbl_consumer = (
    tbl_consumer_raw
    .withColumn("cust_name", f.split(col(string), "\\|").getItem(0))
    .withColumn("address", f.split(col(string), "\\|").getItem(1))
    .withColumn("state", f.split(col(string), "\\|").getItem(2))
    .withColumn("postcode", f.split(col(string), "\\|").getItem(3))
    .withColumn("gender", f.split(col(string), "\\|").getItem(4))
    .withColumn("consumer_id", f.split(col(string), "\\|").getItem(5))
    .drop(string)
)


# Clean merchants table
tbl_merchants = (
    tbl_merchants_raw
    # remove leading (( or [[ and trailing )) or ]]
    .withColumn(
        "tags_clean",
        f.regexp_replace(
            "tags",
            r"^\(\(|^\(\[|^\[\(|^\[\[|\)\)$|\]\)$|\)\]$|\]\]$",
            ""
        )
    )
    # split on `), (` or `], [`
    .withColumn("tags_array", f.split("tags_clean", r"\)\s*,\s*\(|\]\s*,\s*\["))
    # extract each element
    .withColumn("biz_tags", f.lower(f.col("tags_array")[0]))
    .withColumn("rev_band", f.col("tags_array")[1])
    .withColumn("take_rate", f.regexp_extract(f.col("tags_array")[2], r"take rate:\s*([0-9.]+)", 1)
    )
    .drop("tags", "tags_clean", "tags_array")
)

tbl_merchants=tbl_merchants.withColumn("biz_tags", f.regexp_replace("biz_tags", "  ", " "))

In [ ]:
#joining transactions and merchants datasets
merchant_transactions=transactions.join(tbl_merchants, on='merchant_abn', how='left')
print(merchant_transactions.shape)
find_NULL([merchant_transactions])

In [ ]:
merchant_transactions = merchant_transactions.dropna()
print(merchant_transactions.shape)

In [ ]:
merchant_transactions.groupBy('name').count().orderBy("count", ascending=True).show()

In [ ]:
#filter outliers by biz_tag

n = merchant_transactions.count()
# compute the scale factor
scale = m.sqrt(m.log(n)) - 0.5
stats_by_band = (merchant_transactions.groupby('biz_tags')
                                      .agg(f.expr("percentile_approx(dollar_value, 0.25)").alias("Q1"),
                                           f.expr("percentile_approx(dollar_value, 0.75)").alias("Q3")
                ).withColumn("IQR", f.col("Q3") - f.col("Q1"))
                 .withColumn("lower_bound", f.col("Q1") - scale * f.col("IQR"))
                 .withColumn("upper_bound", f.col("Q3") + scale * f.col("IQR"))
                )
stats_by_band = stats_by_band.drop('IQR')

In [ ]:
#remove outliers
merchant_transactions = (
    merchant_transactions
    .join(stats_by_band, on="biz_tags", how="left")
    .filter(
        (col("dollar_value") >= col('lower_bound')) &
        (col("dollar_value") <= col("upper_bound"))
    )
    .select(merchant_transactions["*"])
)

merchant_transactions

In [ ]:
print(merchant_transactions.shape)

In [ ]:
#feature editing
merchant_transactions=merchant_transactions.withColumnRenamed('name', 'business')
merchant_transactions=merchant_transactions.drop('order_id')
merchant_transactions

In [ ]:
#shows min, max, mean by biz tags
merchant_transactions.groupBy("biz_tags").agg(
    f.min("dollar_value").alias("min_value"),
    f.max("dollar_value").alias("max_value"),
    f.mean("dollar_value").alias("mean"),
    (f.max("dollar_value") - f.min("dollar_value")).alias("range")
).orderBy("mean", ascending=False).show()

In [ ]:
#returns distinct business tags
biz_tags_list = merchant_transactions.select("biz_tags").distinct().rdd.flatMap(lambda x: x).collect()
print(biz_tags_list)
print(len(biz_tags_list))

In [ ]:
# Count how many distinct biz_tags each business has
biz_tag_counts = (
    merchant_transactions
    .groupBy("business")
    .agg(f.countDistinct("biz_tags").alias("distinct_tag_count"))
    .orderBy(f.desc("distinct_tag_count"))
)

# Show businesses (with more than one unique tag)
biz_tag_counts

In [ ]:
# Assigning the biz_tags to segments
merchant_transactions = merchant_transactions.withColumn(
    "segment",
    f.when(f.col("biz_tags").isin(
        "watch, clock, and jewelry repair shops",
        "jewelry, watch, clock, and silverware shops",
        "shoe shops",
        "antique shops - sales, repairs, and restoration services",
        "gift, card, novelty, and souvenir shops"
    ), "Fashion, Jewelry & Personal Goods")
   
    .when(f.col("biz_tags").isin(
        "books, periodicals, and newspapers",
        "digital goods: books, movies, music",
        "music shops - musical instruments, pianos, and sheet music",
        "art dealers and galleries",
        "artist supply and craft shops",
        "hobby, toy and game shops",
        "cable, satellite, and other pay television and radio services"
    ), "Arts, Media & Entertainment")
   
    .when(f.col("biz_tags").isin(
        "computers, computer peripheral equipment, and software",
        "computer programming , data processing, and integrated systems design services",
        "telecom",
        "equipment, tool, furniture, and appliance rent al and leasing",
        "stationery, office supplies and printing and writing paper"
    ), "Technology & Professional Services")
   
    .when(f.col("biz_tags").isin(
        "furniture, home furnishings and equipment shops, and manufacturers, except appliances",
        "tent and awning shops",
        "lawn and garden supply outlets, including nurseries",
        "florists supplies, nursery stock, and flowers"
    ), "Home, Garden & Living")
   
    .when(f.col("biz_tags").isin(
        "opticians, optical goods, and eyeglasses",
        "health and beauty spas",
        "bicycle shops - sales and service",
        "motor vehicle supplies and new parts"
    ), "Lifestyle, Health & Recreation")
   
    .otherwise("Other")
)
merchant_transactions

In [ ]:
#write file
merchant_transactions.write.parquet("../data/curated/merchant_transactions", mode="overwrite")

In [ ]:
tbl_consumer=tbl_consumer.drop('address', 'cust_name', 'gender')
#tbl_consumer

In [ ]:
#distinguishes fraud probs
mtc_fraud1=merchant_transactions.join(merch_fraud_prob, on=['merchant_abn', 'order_datetime'], how='left')
mtc_fraud1=mtc_fraud1.withColumnRenamed('fraud_probability', 'merch_fraud_prob')
mtc_fraud=mtc_fraud1.join(con_fraud_prob, on=['user_id', 'order_datetime'], how='left')
mtc_fraud=mtc_fraud.withColumnRenamed('fraud_probability', 'con_fraud_prob')
mtc_fraud

In [ ]:
mtc_fraud

In [ ]:
#aggregates by mean transaction per user per merchant
curated=mtc_fraud.groupBy(['merchant_abn', 'user_id']).agg(
    f.count('*').alias('count'),
    f.mean('dollar_value').alias('mean'),
    f.mean('merch_fraud_prob').alias('merch_fraud_prob'),
    f.mean('con_fraud_prob').alias('con_fraud_prob')
)
curated

In [ ]:
#joins user and consumers
new_curated=curated.join(consumer_user_details, on='user_id', how='left')
new_curated=new_curated.join(tbl_consumer, on='consumer_id', how='left')
new_curated=new_curated.drop('consumer_id')
new_curated

In [ ]:
print(new_curated.shape)

In [ ]:
#writes file
new_curated.write.parquet("../data/curated/agg_by_userbiz", mode="overwrite")

In [ ]:
# Rename columns in df2 for easier handling
income_clean = (
    income_df
    .withColumnRenamed("Statistical Areas Level 2 2021 code", "SA2_CODE_2021")
    .withColumnRenamed("Statistical Areas Level 2 2021 name", "SA2_NAME_2021")
)

# Make sure join keys are the same type
postcodes_df = postcodes_df.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.withColumn("SA2_CODE_2021", col("SA2_CODE_2021").cast("string"))

income_clean = income_clean.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`") != 0)

# Perform join on SA2 code
merged_df = postcodes_df.join(income_clean, on="SA2_CODE_2021", how="left")
merged_df = merged_df.drop(income_clean.SA2_NAME_2021)  # drop df2’s version

missing_count = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNull()).count()
print(missing_count)

# Identify postcodes linked to multiple SA2s 
postcode_sa2_counts = (
    merged_df
    .groupBy("POSTCODE")
    .agg(f.countDistinct("SA2_CODE_2021").alias("num_sa2s"))
)

# Filter postcodes with more than one SA2
multiple_sa2_postcodes = postcode_sa2_counts.filter(f.col("num_sa2s") > 1)
print("Postcodes linked to multiple SA2s:")
multiple_sa2_postcodes.show(10, truncate=False)

# Show total number of postcodes that map to multiple SA2s
total_multi_sa2 = multiple_sa2_postcodes.count()
print(f"Total postcodes linked to multiple SA2s: {total_multi_sa2}")

# Total number of unique postcodes overall
total_postcodes = postcode_sa2_counts.count()
print(f"Total unique postcodes in dataset: {total_postcodes}")

# Dropping NULL values in income column
print(merged_df.count()) # count before dropping NULL values
merged_df = merged_df.filter(col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`").isNotNull() )
print(merged_df.count()) # count after dropping NULL values

result_df = (
    merged_df
    .groupBy(col("POSTCODE").alias("postcode"))
    .agg(
        f.avg(
            col("`Personal income: Median total income (excl. Government pensions and allowances) ($) (Data year: 2020)`")
        ).alias("median_total_income_2020")
    )
)
result_df

In [ ]:
result_df = result_df.withColumn(
    "income_bin",
    f.when(f.col("median_total_income_2020") < 30000, "<30k")
     .when((f.col("median_total_income_2020") >= 30000) & (f.col("median_total_income_2020") < 40000), "30-40k")
     .when((f.col("median_total_income_2020") >= 40000) & (f.col("median_total_income_2020") < 50000), "40-50k")
     .when((f.col("median_total_income_2020") >= 50000) & (f.col("median_total_income_2020") < 60000), "50-60k")
     .when((f.col("median_total_income_2020") >= 60000) & (f.col("median_total_income_2020") < 70000), "60-70k")
     .when((f.col("median_total_income_2020") >= 70000) & (f.col("median_total_income_2020") < 80000), "70-80k")
     .otherwise("80k+")
)

result_df.show(100, truncate=False)
result_df.printSchema()


In [ ]:
result_df.write.csv("../data/curated/merged_postcode_income.csv", header=True, mode="overwrite")